# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 10

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. В чем практическая разница между обращением к данным через `.loc[]` и `.iloc[]` в библиотеке Pandas?
```python
val_a = df.loc[0, 'nickname']
val_b = df.iloc[0, 1]
```
По какому принципу находит элемент каждый из этих методов?

**Ответ:**
* `.loc[]` осуществляет доступ по **пользовательским именам и меткам** строк и колонок (label-based: название столбца 'nickname', исходная метка строки 0).
* `.iloc[]` осуществляет доступ строго по **целочисленным порядковым номерам позиций** (integer position-based: 0-я строка, 1-й по счету столбец).

---

#### 1.2. Аналитик выполнил две команды:
```python
cnt = df['city'].nunique()
lst = df['city'].unique()
```
Чем принципиально различаются результаты в переменных `cnt` и `lst`? Что возвращает число, а что — массив самих названий?

**Ответ:**
* `cnt = df['city'].nunique()` возвращает одно целое **число** — количество различных уникальных городов.
* `lst = df['city'].unique()` возвращает **массив/список** с самими уникальными названиями этих городов.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["effective_hourly_rate"] = (df["total_paid"] / df["duration_hours"]).round(2)
```

**Ответ:**
Рассчитывает фактическую стоимость одного игрового часа для клиента с учетом всех скидок и округляет до копеек.

---

#### 1.4. Зачем после группировки данных в Pandas аналитики вызывают метод `.reset_index()`?
```python
summary = df.groupby('category')['total_amount'].sum().reset_index()
```
Как изменится структура таблицы `summary` при добавлении `.reset_index()`?

**Ответ:**
Метод `.reset_index()` переносит столбец группировки (`category`) из индекса обратно в обычные столбцы данных и возвращает таблице стандартный целочисленный индекс строк от 0. Без него `category` осталась бы индексом.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `tournaments.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("tournaments.csv").parent if os.path.exists("tournaments.csv") else pathlib.Path("../variant_10")
df_audit = pd.read_csv(data_dir / "tournaments.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Средний взнос на турнир: 525.00")

Количество строк в таблице: 8
Столбцы датафрейма: ['tournament_id', 'tournament_name', 'game_title', 'prize_fund', 'entry_fee', 'start_date', 'min_rank']
Средний взнос на турнир: 525.00

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `game_sessions.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path(f"../variant_10")
df_data = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

summary = df_data.groupby("game_title").agg({
    "total_paid": "sum",
    "duration_hours": "mean"
}).reset_index()

summary["duration_hours"] = summary["duration_hours"].round(2)
summary = summary.sort_values(by="total_paid", ascending=False).reset_index(drop=True)
display(summary)

,game_title,total_paid,duration_hours
0,Valorant,26919,2.33
1,Atomic Heart,26866,2.11
2,Apex Legends,25255,2.10
3,Counter-Strike 2,25238,2.31
4,Dota 2,23551,2.01
5,GTA V,23363,2.08
6,Warzone,23010,2.07
7,PUBG,21148,2.21
8,Rust,19396,2.23
9,Cyberpunk 2077,17696,2.04


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `game_sessions` колонка `member_id` является внешним ключом (`FOREIGN KEY`), ссылающимся на таблицу `members`.
Зачем нужна эта связь и что произойдет, если попытаться добавить игровую сессию для несуществующего клиента с `member_id = 99999`?

**Ответ:**
Внешний ключ связывает дочернюю таблицу (`game_sessions`) с родительской (`members`) и обеспечивает **ссылочную целостность**. СУБД отклонит вставку сессии с несуществующим клиентом (`FOREIGN KEY constraint failed`), предотвращая появление «потерянных» записей.

---

#### 3.2. Администратор киберклуба случайно выполнил запрос `DELETE FROM game_sessions;` вместо удаления одной устаревшей сессии.
Если запрос выполнялся внутри открытой транзакции, какой командой можно отменить удаление до того, как изменения зафиксируются на диске?

**Ответ:**
Командой `ROLLBACK`. Она отменяет все операции текущей открытой транзакции и возвращает данные в исходное состояние (до вызова команды `COMMIT`).

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT min_rank, COUNT(*) AS tourn_cnt, SUM(prize_fund) AS total_prizes FROM tournaments GROUP BY min_rank ORDER BY total_prizes DESC;
```

**Ответ:**
Запрос группирует турниры по минимальному рангу участников (`min_rank`) и рассчитывает количество турниров и суммарный призовой фонд, упорядочивая по убыванию призовых.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_10")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT session_id, member_id, game_title, duration_hours, discount_promo, total_paid
FROM game_sessions
WHERE game_title = 'Dota 2' AND duration_hours >= 2.5 AND discount_promo = 0
ORDER BY duration_hours DESC, total_paid DESC
LIMIT 5;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,session_id,member_id,game_title,duration_hours,discount_promo,total_paid
0,20225,53,Dota 2,6.0,0,1500.0
1,20486,53,Dota 2,6.0,0,1500.0
2,20071,56,Dota 2,6.0,0,1020.0
3,20376,98,Dota 2,5.0,0,2100.0
4,20456,57,Dota 2,4.0,0,1000.0


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT client_rating, COUNT(*) AS sessions_count, ROUND(SUM(total_paid), 2) AS total_revenue, ROUND(AVG(duration_hours), 2) AS avg_duration
FROM game_sessions
WHERE client_rating IS NOT NULL
GROUP BY client_rating
HAVING COUNT(*) >= 30
ORDER BY client_rating DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,client_rating,sessions_count,total_revenue,avg_duration
0,5.0,170,83949.0,2.31
1,4.0,167,75146.0,2.06
2,3.0,58,26130.0,2.03


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT m.member_id, m.nickname, COUNT(s.session_id) AS sessions_count, ROUND(SUM(s.total_paid), 2) AS total_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
WHERE m.city = 'Санкт-Петербург'
GROUP BY m.member_id, m.nickname
ORDER BY total_spent DESC
LIMIT 8;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,member_id,nickname,sessions_count,total_spent
0,1,Rogue_593,8,4360.0
1,117,Phoenix_222,9,4303.0
2,47,Grizzly_718,7,4300.0
3,42,Phantom_722,9,4277.0
4,40,Raven_271,8,4035.0
5,48,Slayer_348,9,3930.0
6,19,Phoenix_253,7,3565.0
7,57,Ronin_999,6,3510.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT pc_id, pc_number, zone_name, hourly_rate, ram_gb
FROM hardware_pcs
WHERE ram_gb > (SELECT AVG(ram_gb) FROM hardware_pcs)
ORDER BY ram_gb DESC, pc_number ASC;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,pc_id,pc_number,zone_name,hourly_rate,ram_gb
0,23,PC-23,VIP,400.0,64
1,24,PC-24,VIP,420.0,64
2,25,PC-25,VIP,380.0,64
3,26,PC-26,VIP,400.0,64
4,27,PC-27,VIP,400.0,64
5,15,PC-15,Bootcamp,250.0,32
6,16,PC-16,Bootcamp,250.0,32
7,17,PC-17,Bootcamp,250.0,32
8,18,PC-18,Bootcamp,250.0,32
9,19,PC-19,Bootcamp,230.0,32


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Крупные заказы в баре клуба

**Найденные ошибки:**
1. Ошибка ключа соединения: условие `ON m.member_id = b.order_id` связывает идентификатор клиента с номером заказа вместо `b.member_id`.

In [9]:
query_51_fixed = '''
SELECT 
    m.nickname,
    m.city,
    b.item_name,
    b.total_amount
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
WHERE b.total_amount > 400
ORDER BY b.total_amount DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,nickname,city,item_name,total_amount
0,Sentinel_224,Новосибирск,Мерч-футболка CyberClub,4500.0
1,Striker_272,Москва,Мерч-футболка CyberClub,4500.0
2,Blaze_356,Москва,Мерч-футболка CyberClub,4500.0
3,Dragon_483,Москва,Мерч-футболка CyberClub,3000.0
4,Falcon_534,Санкт-Петербург,Мерч-футболка CyberClub,3000.0
5,CyberKnight_173,Екатеринбург,Мерч-футболка CyberClub,3000.0
6,Phantom_819,Москва,Мерч-футболка CyberClub,3000.0
7,Kaiser_771,Москва,Мерч-футболка CyberClub,3000.0
8,Shadow_779,Новосибирск,Мерч-футболка CyberClub,3000.0
9,Valkyrie_745,Санкт-Петербург,Мерч-футболка CyberClub,3000.0


#### 5.2. Суммарные заказы клиентов в баре

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = b.order_id` связывает клиента с номером заказа вместо `b.member_id`.
2. Неполный `GROUP BY`: столбец `m.nickname` указан в `SELECT`, но отсутствует в списке полей группировки.

In [10]:
query_52_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    SUM(b.total_amount) AS bar_spent
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
GROUP BY m.member_id, m.nickname
HAVING SUM(b.total_amount) > 1000
ORDER BY bar_spent DESC
LIMIT 10;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,member_id,nickname,bar_spent
0,63,Phantom_819,8620.0
1,100,Blaze_356,6720.0
2,121,Striker_272,6220.0
3,114,Tempest_629,5580.0
4,69,Sentinel_224,5330.0
5,64,Dragon_483,4680.0
6,101,Blaze_958,4640.0
7,97,CyberKnight_173,3840.0
8,129,Kaiser_771,3760.0
9,58,Titan_614,3740.0


#### 5.3. Сравнение призовых фондов турниров со средним фондом

**Найденные ошибки:**
1. Неполный `GROUP BY`: поле `min_rank` указано в `SELECT`, но не включено в список группировки.
2. Порядок предложений и alias: секция `WHERE` написана после `GROUP BY` и использует псевдоним `avg_prize` вместо `HAVING AVG(prize_fund)`.
3. Многострочный подзапрос: наличие `GROUP BY game_title` в подзапросе возвращает несколько строк вместо одного скалярного значения.

In [11]:
query_53_fixed = '''
SELECT 
    game_title,
    min_rank,
    ROUND(AVG(prize_fund), 2) AS avg_prize
FROM tournaments
GROUP BY game_title, min_rank
HAVING AVG(prize_fund) > (
    SELECT AVG(prize_fund) 
    FROM tournaments
)
ORDER BY avg_prize DESC;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,game_title,min_rank,avg_prize
0,Counter-Strike 2,Gold,120000.0
1,Dota 2,Silver,75000.0
